
# DATATHON 2026 — Plotly EDA Notebook (19 biểu đồ, đơn vị VNĐ)

Notebook này được viết theo đúng yêu cầu:
- **19 nội dung visualize** như outline bạn cung cấp
- Biểu đồ **to, rõ ràng, sạch**
- **Chú thích đầy đủ**, title và axis title rõ
- Hạn chế chồng lấn
- **Các line mảnh, nhỏ** để dễ nhìn
- Khi một cell có nhiều biểu đồ, các subplot được **xếp theo hàng (rows)**, **không gộp theo cột**
- **Đơn vị tiền tệ: VNĐ**

## Cách dùng
1. Đặt các file CSV của cuộc thi vào cùng thư mục notebook, hoặc trong:
   - `./data/`
   - `./dataset/`
   - `/mnt/data/`
   - `/mnt/data/data/`
2. Giữ nguyên tên file gốc:
   - `products.csv`, `customers.csv`, `promotions.csv`, `geography.csv`
   - `orders.csv`, `order_items.csv`, `payments.csv`, `shipments.csv`, `returns.csv`, `reviews.csv`
   - `sales.csv`, `inventory.csv`, `web_traffic.csv`
3. Run notebook từ trên xuống dưới.

## Ghi chú
- Notebook được viết để **tự tìm file CSV** trong các thư mục phổ biến.
- Vì trong môi trường hiện tại chưa có toàn bộ CSV gốc, notebook được tạo ở dạng **template hoàn chỉnh sẵn chạy**.
- Với các biểu đồ doanh thu / lợi nhuận từ transaction-level, notebook dùng:
  - `line_revenue = quantity * unit_price`
  - `line_cogs = quantity * cogs`
  - `gross_profit = line_revenue - line_cogs`
- Promo được định nghĩa là dòng có `promo_id` hoặc `promo_id_2` khác null.


In [1]:

# === 0. Imports ===
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)

PLOTLY_TEMPLATE = "plotly_white"
COLOR_SEQ = px.colors.qualitative.Safe

px.defaults.template = PLOTLY_TEMPLATE
px.defaults.color_discrete_sequence = COLOR_SEQ
px.defaults.width = 1500
px.defaults.height = 760


In [2]:

# === 1. Helper functions ===

MONTH_ORDER = list(range(1, 13))
MONTH_NAME_MAP = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

THIN_LINE = 1.4
MEDIUM_LINE = 1.8

def vnd_text(x):
    if pd.isna(x):
        return "NA"
    return f"{x:,.0f} VNĐ"

def monthly_period(series):
    return pd.to_datetime(series).dt.to_period("M").dt.to_timestamp()

def weekly_period(series):
    return pd.to_datetime(series).dt.to_period("W").dt.start_time

def require_tables(dfs, names):
    missing = [n for n in names if n not in dfs or dfs[n] is None]
    if missing:
        raise ValueError(f"Thiếu bảng dữ liệu: {missing}")

def style_figure(fig, title=None, height=760, legend_orientation="v", legend_x=1.02, legend_y=1.0):
    if title is not None:
        fig.update_layout(
            title=dict(text=title, x=0.01, xanchor="left")
        )
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        width=1500,
        height=height,
        font=dict(size=14),
        title_font=dict(size=24),
        hoverlabel=dict(font_size=13),
        hovermode="x unified",
        margin=dict(l=80, r=170, t=90, b=80),
        legend=dict(
            orientation=legend_orientation,
            x=legend_x,
            y=legend_y,
            xanchor="left",
            yanchor="top",
            font=dict(size=12),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.08)",
            borderwidth=1,
        ),
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        automargin=True,
        title_font=dict(size=16),
        tickfont=dict(size=12),
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        automargin=True,
        title_font=dict(size=16),
        tickfont=dict(size=12),
        separatethousands=True,
    )
    return fig

def add_vnd_axis_title(fig, axis="y", label="Giá trị (VNĐ)"):
    if axis == "y":
        fig.update_yaxes(title_text=label, tickformat=",.0f")
    else:
        fig.update_xaxes(title_text=label, tickformat=",.0f")
    return fig

def ordered_month_cols(cols):
    wanted = [MONTH_NAME_MAP[m] for m in MONTH_ORDER]
    return [c for c in wanted if c in cols]

def load_csv_safely(path, parse_dates=None):
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=parse_dates or [])

print("Helpers loaded.")


Helpers loaded.


In [3]:

# === 2. Auto-load CSV files ===

EXPECTED_FILES = {
    "products": "products.csv",
    "customers": "customers.csv",
    "promotions": "promotions.csv",
    "geography": "geography.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "payments": "payments.csv",
    "shipments": "shipments.csv",
    "returns": "returns.csv",
    "reviews": "reviews.csv",
    "sales": "sales.csv",
    "inventory": "inventory.csv",
    "web_traffic": "web_traffic.csv",
}

DATE_COLUMNS = {
    "customers": ["signup_date"],
    "promotions": ["start_date", "end_date"],
    "orders": ["order_date"],
    "shipments": ["ship_date", "delivery_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "sales": ["Date"],
    "inventory": ["snapshot_date"],
    "web_traffic": ["date"],
}

SEARCH_DIRS = [
    Path.cwd(),
    Path.cwd() / "data",
    Path.cwd() / "dataset",
    Path("/mnt/data"),
    Path("/mnt/data/data"),
    Path("/mnt/data/dataset"),
]

def find_file(filename):
    for d in SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    for d in SEARCH_DIRS:
        if d.exists():
            matches = list(d.rglob(filename))
            if matches:
                return matches[0]
    return None

dfs = {}
for name, fn in EXPECTED_FILES.items():
    fp = find_file(fn)
    if fp is None:
        dfs[name] = None
        print(f"[WARN] Không tìm thấy {fn}")
    else:
        dfs[name] = pd.read_csv(fp, parse_dates=DATE_COLUMNS.get(name, []))
        print(f"[OK] {name:12s} -> {fp} | shape={dfs[name].shape}")

print("\nLoaded tables:")
print([k for k, v in dfs.items() if v is not None])


[OK] products     -> f:\Datathon 2026\dataset\products.csv | shape=(2412, 8)
[OK] customers    -> f:\Datathon 2026\dataset\customers.csv | shape=(121930, 7)
[OK] promotions   -> f:\Datathon 2026\dataset\promotions.csv | shape=(50, 10)
[OK] geography    -> f:\Datathon 2026\dataset\geography.csv | shape=(39948, 4)
[OK] orders       -> f:\Datathon 2026\dataset\orders.csv | shape=(646945, 8)
[OK] order_items  -> f:\Datathon 2026\dataset\order_items.csv | shape=(714669, 7)
[OK] payments     -> f:\Datathon 2026\dataset\payments.csv | shape=(646945, 4)
[OK] shipments    -> f:\Datathon 2026\dataset\shipments.csv | shape=(566067, 4)
[OK] returns      -> f:\Datathon 2026\dataset\returns.csv | shape=(39939, 7)
[OK] reviews      -> f:\Datathon 2026\dataset\reviews.csv | shape=(113551, 7)
[OK] sales        -> f:\Datathon 2026\dataset\sales.csv | shape=(3833, 3)
[OK] inventory    -> f:\Datathon 2026\dataset\inventory.csv | shape=(60247, 17)
[OK] web_traffic  -> f:\Datathon 2026\dataset\web_traffic.c

In [4]:

# === 3. Preprocess core analytical tables ===

# 3.1 sales_daily
if dfs.get("sales") is not None:
    sales_daily = dfs["sales"].copy()
    sales_daily["Date"] = pd.to_datetime(sales_daily["Date"])
    sales_daily = sales_daily.sort_values("Date")
    sales_daily["Gross_Profit"] = sales_daily["Revenue"] - sales_daily["COGS"]
    sales_daily["Gross_Margin"] = np.where(
        sales_daily["Revenue"].ne(0),
        sales_daily["Gross_Profit"] / sales_daily["Revenue"],
        np.nan
    )
else:
    require_tables(dfs, ["orders", "order_items", "products"])
    tmp = (
        dfs["order_items"]
        .merge(dfs["orders"][["order_id", "order_date"]], on="order_id", how="left")
        .merge(dfs["products"][["product_id", "cogs"]], on="product_id", how="left")
    )
    tmp["Date"] = pd.to_datetime(tmp["order_date"])
    tmp["Revenue"] = tmp["quantity"] * tmp["unit_price"]
    tmp["COGS"] = tmp["quantity"] * tmp["cogs"]
    sales_daily = tmp.groupby("Date", as_index=False)[["Revenue", "COGS"]].sum()
    sales_daily["Gross_Profit"] = sales_daily["Revenue"] - sales_daily["COGS"]
    sales_daily["Gross_Margin"] = np.where(
        sales_daily["Revenue"].ne(0),
        sales_daily["Gross_Profit"] / sales_daily["Revenue"],
        np.nan
    )

sales_daily["year"] = sales_daily["Date"].dt.year
sales_daily["month"] = sales_daily["Date"].dt.month
sales_daily["month_name"] = sales_daily["month"].map(MONTH_NAME_MAP)
sales_daily["week_start"] = weekly_period(sales_daily["Date"])
sales_daily["month_start"] = monthly_period(sales_daily["Date"])

sales_weekly = sales_daily.groupby("week_start", as_index=False)[["Revenue", "COGS", "Gross_Profit"]].sum()
sales_weekly["Gross_Margin"] = np.where(sales_weekly["Revenue"].ne(0), sales_weekly["Gross_Profit"] / sales_weekly["Revenue"], np.nan)

sales_monthly = sales_daily.groupby("month_start", as_index=False)[["Revenue", "COGS", "Gross_Profit"]].sum()
sales_monthly["Gross_Margin"] = np.where(sales_monthly["Revenue"].ne(0), sales_monthly["Gross_Profit"] / sales_monthly["Revenue"], np.nan)
sales_monthly["year"] = sales_monthly["month_start"].dt.year
sales_monthly["month"] = sales_monthly["month_start"].dt.month
sales_monthly["month_name"] = sales_monthly["month"].map(MONTH_NAME_MAP)

# 3.2 transaction-wide table
require_tables(dfs, ["orders", "order_items", "products"])
orders = dfs["orders"].copy()
order_items = dfs["order_items"].copy()
products = dfs["products"].copy()

line_items = (
    order_items
    .merge(
        orders[["order_id", "order_date", "customer_id", "zip", "order_status", "payment_method", "device_type", "order_source"]],
        on="order_id",
        how="left"
    )
    .merge(
        products[["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"]],
        on="product_id",
        how="left"
    )
)

if dfs.get("customers") is not None:
    customers = dfs["customers"][["customer_id", "age_group", "gender", "acquisition_channel", "city"]].copy()
    line_items = line_items.merge(customers, on="customer_id", how="left")

if dfs.get("geography") is not None:
    geography = dfs["geography"][["zip", "region", "city", "district"]].copy()
    line_items = line_items.merge(geography, on="zip", how="left", suffixes=("", "_geo"))

line_items["order_date"] = pd.to_datetime(line_items["order_date"])
line_items["Revenue"] = line_items["quantity"] * line_items["unit_price"]
line_items["COGS"] = line_items["quantity"] * line_items["cogs"]
line_items["Gross_Profit"] = line_items["Revenue"] - line_items["COGS"]
line_items["Gross_Margin"] = np.where(line_items["Revenue"].ne(0), line_items["Gross_Profit"] / line_items["Revenue"], np.nan)
line_items["promo_flag"] = line_items[["promo_id", "promo_id_2"]].notna().any(axis=1)
line_items["month_start"] = monthly_period(line_items["order_date"])
line_items["week_start"] = weekly_period(line_items["order_date"])
line_items["year"] = line_items["order_date"].dt.year
line_items["month"] = line_items["order_date"].dt.month
line_items["month_name"] = line_items["month"].map(MONTH_NAME_MAP)

product_perf = (
    line_items
    .groupby(["product_id", "product_name", "category", "segment", "price"], as_index=False)
    .agg(
        units_sold=("quantity", "sum"),
        Revenue=("Revenue", "sum"),
        COGS=("COGS", "sum"),
        Gross_Profit=("Gross_Profit", "sum")
    )
)
product_perf["Gross_Margin"] = np.where(product_perf["Revenue"].ne(0), product_perf["Gross_Profit"] / product_perf["Revenue"], np.nan)

if dfs.get("inventory") is not None:
    inventory = dfs["inventory"].copy()
    inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"])
else:
    inventory = None

if dfs.get("returns") is not None:
    returns_df = dfs["returns"].copy()
    returns_df = returns_df.merge(
        products[["product_id", "category", "segment", "product_name"]],
        on="product_id",
        how="left"
    )
else:
    returns_df = None

if dfs.get("reviews") is not None:
    reviews_df = dfs["reviews"].copy().merge(
        products[["product_id", "category", "segment"]],
        on="product_id",
        how="left"
    )
else:
    reviews_df = None

if dfs.get("promotions") is not None:
    promotions = dfs["promotions"].copy()
else:
    promotions = None

if dfs.get("web_traffic") is not None:
    web = dfs["web_traffic"].copy()
    web["date"] = pd.to_datetime(web["date"])
else:
    web = None

print("Preprocessing done.")
print("sales_daily:", sales_daily.shape)
print("line_items :", line_items.shape)
print("product_perf:", product_perf.shape)


Preprocessing done.
sales_daily: (3833, 10)
line_items : (714669, 38)
product_perf: (1598, 10)


## 1) Revenue theo ngày / tuần / tháng

In [27]:

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=("Revenue theo ngày", "Revenue theo tuần", "Revenue theo tháng")
)

fig.add_trace(
    go.Scatter(
        x=sales_daily["Date"], y=sales_daily["Revenue"],
        mode="lines", name="Daily Revenue",
        line=dict(width=THIN_LINE),
        hovertemplate="Date: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=sales_weekly["week_start"], y=sales_weekly["Revenue"],
        mode="lines", name="Weekly Revenue",
        line=dict(width=THIN_LINE),
        hovertemplate="Week: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=sales_monthly["month_start"], y=sales_monthly["Revenue"],
        mode="lines+markers", name="Monthly Revenue",
        line=dict(width=MEDIUM_LINE),
        marker=dict(size=5),
        hovertemplate="Month: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ),
    row=3, col=1
)

fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=1, col=1)
fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=2, col=1)
fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=3, col=1)
fig.update_xaxes(title_text="Date", row=3, col=1)

style_figure(fig, "1. Revenue theo ngày / tuần / tháng", height=1200)
fig.show()


## 2) Revenue / COGS / Gross Profit theo thời gian

In [6]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sales_monthly["month_start"], y=sales_monthly["Revenue"],
    mode="lines+markers", name="Revenue",
    line=dict(width=MEDIUM_LINE),
    marker=dict(size=4),
    hovertemplate="Month: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=sales_monthly["month_start"], y=sales_monthly["COGS"],
    mode="lines+markers", name="COGS",
    line=dict(width=MEDIUM_LINE),
    marker=dict(size=4),
    hovertemplate="Month: %{x}<br>COGS: %{y:,.0f} VNĐ<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=sales_monthly["month_start"], y=sales_monthly["Gross_Profit"],
    mode="lines+markers", name="Gross Profit",
    line=dict(width=MEDIUM_LINE),
    marker=dict(size=4),
    hovertemplate="Month: %{x}<br>Gross Profit: %{y:,.0f} VNĐ<extra></extra>"
))

style_figure(fig, "2. Revenue / COGS / Gross Profit theo thời gian", height=760)
fig.update_xaxes(title_text="Thời gian")
fig.update_yaxes(title_text="Giá trị (VNĐ)", tickformat=",.0f")
fig.show()


## 3) Gross Margin theo tháng / năm

In [7]:

gm_month = (
    sales_monthly.groupby("month", as_index=False)["Gross_Margin"]
    .mean()
    .sort_values("month")
)
gm_month["month_name"] = gm_month["month"].map(MONTH_NAME_MAP)

gm_year = (
    sales_daily.groupby("year", as_index=False)[["Revenue", "Gross_Profit"]]
    .sum()
)
gm_year["Gross_Margin"] = np.where(gm_year["Revenue"].ne(0), gm_year["Gross_Profit"] / gm_year["Revenue"], np.nan)

fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.15,
    subplot_titles=("Gross Margin trung bình theo tháng", "Gross Margin theo năm")
)

fig.add_trace(
    go.Bar(
        x=gm_month["month_name"], y=gm_month["Gross_Margin"],
        text=gm_month["Gross_Margin"].map(lambda x: f"{x:.1%}"),
        textposition="outside",
        name="Avg GM by Month"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=gm_year["year"], y=gm_year["Gross_Margin"],
        mode="lines+markers+text",
        text=gm_year["Gross_Margin"].map(lambda x: f"{x:.1%}"),
        textposition="top center",
        name="GM by Year",
        line=dict(width=MEDIUM_LINE),
        marker=dict(size=6),
        hovertemplate="Year: %{x}<br>Gross Margin: %{y:.2%}<extra></extra>"
    ),
    row=2, col=1
)

fig.update_yaxes(title_text="Gross Margin", tickformat=".1%", row=1, col=1)
fig.update_yaxes(title_text="Gross Margin", tickformat=".1%", row=2, col=1)
fig.update_xaxes(title_text="Tháng", row=1, col=1)
fig.update_xaxes(title_text="Năm", row=2, col=1)

style_figure(fig, "3. Gross Margin theo tháng / năm", height=1100)
fig.show()


## 4) Heatmap doanh thu theo Month × Year

In [8]:

heat_df = sales_daily.groupby(["year", "month"], as_index=False)["Revenue"].sum()
heat_df["month_name"] = heat_df["month"].map(MONTH_NAME_MAP)
pivot = heat_df.pivot(index="year", columns="month_name", values="Revenue")
pivot = pivot[ordered_month_cols(pivot.columns)]

fig = px.imshow(
    pivot,
    aspect="auto",
    text_auto=".2s",
    color_continuous_scale="Blues"
)
style_figure(fig, "4. Heatmap doanh thu theo Month × Year", height=700)
fig.update_xaxes(title_text="Month")
fig.update_yaxes(title_text="Year")
fig.show()


## 5) Doanh thu theo Category / Segment

In [9]:

rev_cat = line_items.groupby("category", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=True)
rev_seg = line_items.groupby("segment", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=True)

fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.12,
    subplot_titles=("Revenue theo Category", "Revenue theo Segment")
)

fig.add_trace(
    go.Bar(
        x=rev_cat["Revenue"], y=rev_cat["category"],
        orientation="h",
        text=rev_cat["Revenue"].map(vnd_text),
        textposition="outside",
        name="Category Revenue"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=rev_seg["Revenue"], y=rev_seg["segment"],
        orientation="h",
        text=rev_seg["Revenue"].map(vnd_text),
        textposition="outside",
        name="Segment Revenue"
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=1, col=1)
fig.update_xaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=2, col=1)
fig.update_yaxes(title_text="Category", row=1, col=1)
fig.update_yaxes(title_text="Segment", row=2, col=1)

style_figure(fig, "5. Doanh thu theo Category / Segment", height=1150)
fig.show()


## 6) Gross Margin theo Segment

In [10]:

gm_seg = line_items.groupby("segment", as_index=False)[["Revenue", "Gross_Profit"]].sum()
gm_seg["Gross_Margin"] = np.where(gm_seg["Revenue"].ne(0), gm_seg["Gross_Profit"] / gm_seg["Revenue"], np.nan)
gm_seg = gm_seg.sort_values("Gross_Margin", ascending=False)

fig = px.bar(
    gm_seg,
    x="segment",
    y="Gross_Margin",
    color="segment",
    text=gm_seg["Gross_Margin"].map(lambda x: f"{x:.1%}")
)

style_figure(fig, "6. Gross Margin theo Segment", height=700)
fig.update_xaxes(title_text="Segment", tickangle=0)
fig.update_yaxes(title_text="Gross Margin", tickformat=".1%")
fig.show()


## 7) Scatter plot: price vs units sold vs revenue

In [11]:

fig = px.scatter(
    product_perf,
    x="price",
    y="units_sold",
    size="Revenue",
    color="segment",
    hover_name="product_name",
    hover_data={
        "Revenue": ":,.0f",
        "price": ":,.0f",
        "units_sold": ":,.0f"
    },
    size_max=42
)

style_figure(fig, "7. Price vs Units Sold vs Revenue", height=780)
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>Price: %{x:,.0f} VNĐ<br>Units Sold: %{y:,.0f}<br>Revenue: %{marker.size:,.0f} VNĐ<extra></extra>",
    marker=dict(line=dict(width=0.5, color="rgba(0,0,0,0.2)"))
)
fig.update_xaxes(title_text="Price (VNĐ)", tickformat=",.0f")
fig.update_yaxes(title_text="Units Sold")
fig.show()


## 8) Promo-adoption rate theo Category

In [12]:

promo_cat = (
    line_items.groupby("category", as_index=False)
    .agg(lines=("order_id", "size"), promo_lines=("promo_flag", "sum"))
)
promo_cat["promo_adoption_rate"] = promo_cat["promo_lines"] / promo_cat["lines"]
promo_cat = promo_cat.sort_values("promo_adoption_rate", ascending=False)

fig = px.bar(
    promo_cat,
    x="category",
    y="promo_adoption_rate",
    color="category",
    text=promo_cat["promo_adoption_rate"].map(lambda x: f"{x:.1%}")
)

style_figure(fig, "8. Promo-adoption rate theo Category", height=700)
fig.update_xaxes(title_text="Category")
fig.update_yaxes(title_text="Promo Adoption Rate", tickformat=".1%")
fig.show()


## 9) Revenue theo thời gian: promo vs non-promo

In [13]:

promo_time = line_items.groupby(["month_start", "promo_flag"], as_index=False)["Revenue"].sum()
promo_time["promo_group"] = np.where(promo_time["promo_flag"], "Promo", "Non-Promo")

fig = px.area(
    promo_time,
    x="month_start",
    y="Revenue",
    color="promo_group",
    line_group="promo_group"
)
style_figure(fig, "9. Revenue theo thời gian: promo vs non-promo", height=760)
fig.update_xaxes(title_text="Thời gian")
fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f")
fig.show()


## 10) Event-study cho 10 campaign gần đây nhất bằng line chart

In [14]:

if promotions is None:
    print("Thiếu promotions.csv -> bỏ qua chart 10")
else:
    promo_long_frames = []
    for c in ["promo_id", "promo_id_2"]:
        tmp = line_items[["order_id", "order_date", "Revenue", c]].copy()
        tmp = tmp.rename(columns={c: "promo_id"})
        tmp = tmp[tmp["promo_id"].notna()]
        promo_long_frames.append(tmp)

    promo_long = pd.concat(promo_long_frames, ignore_index=True).drop_duplicates()
    promo_long = promo_long.merge(
        promotions[["promo_id", "promo_name", "start_date", "end_date"]],
        on="promo_id",
        how="left"
    )
    promo_long["start_date"] = pd.to_datetime(promo_long["start_date"])
    promo_long["end_date"] = pd.to_datetime(promo_long["end_date"])

    recent_10 = (
        promo_long[["promo_id", "promo_name", "start_date", "end_date"]]
        .drop_duplicates()
        .sort_values("start_date", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    fig = make_subplots(
        rows=len(recent_10), cols=1,
        shared_xaxes=False,
        vertical_spacing=0.02,
        subplot_titles=[
            f"{row['promo_name'] if pd.notna(row['promo_name']) else row['promo_id']}"
            for _, row in recent_10.iterrows()
        ]
    )

    for idx, row in recent_10.iterrows():
        s = row["start_date"]
        e = row["end_date"]
        promo_days = (e - s).days if pd.notna(e) and pd.notna(s) else 0

        event_window = sales_daily[
            (sales_daily["Date"] >= s - pd.Timedelta(days=14)) &
            (sales_daily["Date"] <= e + pd.Timedelta(days=14))
        ].copy()

        event_window["event_day"] = (event_window["Date"] - s).dt.days

        fig.add_trace(
            go.Scatter(
                x=event_window["event_day"],
                y=event_window["Revenue"],
                mode="lines+markers",
                line=dict(width=THIN_LINE),
                marker=dict(size=4),
                name=str(row["promo_id"]),
                showlegend=False,
                hovertemplate="Event day: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
            ),
            row=idx + 1, col=1
        )

        fig.add_vrect(
            x0=0,
            x1=max(promo_days, 0),
            fillcolor="rgba(255,165,0,0.15)",
            line_width=0,
            row=idx + 1,
            col=1
        )

    style_figure(fig, "10. Event-study cho 10 campaign gần đây nhất", height=max(2200, 220 * len(recent_10)))
    fig.update_xaxes(title_text="Số ngày tương đối so với ngày bắt đầu campaign")
    fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f")
    fig.show()


## 11) Return quantity và refund amount theo Category × Return Reason

In [15]:

if returns_df is None:
    print("Thiếu returns.csv -> bỏ qua chart 11")
else:
    qty_pivot = (
        returns_df.groupby(["category", "return_reason"], as_index=False)["return_quantity"]
        .sum()
        .pivot(index="category", columns="return_reason", values="return_quantity")
        .fillna(0)
    )

    refund_pivot = (
        returns_df.groupby(["category", "return_reason"], as_index=False)["refund_amount"]
        .sum()
        .pivot(index="category", columns="return_reason", values="refund_amount")
        .fillna(0)
    )

    fig = make_subplots(
        rows=2, cols=1,
        vertical_spacing=0.16,
        subplot_titles=(
            "Return Quantity theo Category × Return Reason",
            "Refund Amount theo Category × Return Reason"
        )
    )

    fig.add_trace(
        go.Heatmap(
            z=qty_pivot.values,
            x=list(qty_pivot.columns),
            y=list(qty_pivot.index),
            text=qty_pivot.values,
            texttemplate="%{text:.0f}",
            colorscale="Oranges",
            colorbar=dict(title="Qty", len=0.35, y=0.83)
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Heatmap(
            z=refund_pivot.values,
            x=list(refund_pivot.columns),
            y=list(refund_pivot.index),
            text=refund_pivot.values,
            texttemplate="%{text:,.0f}",
            colorscale="Reds",
            colorbar=dict(title="VNĐ", len=0.35, y=0.17)
        ),
        row=2, col=1
    )

    style_figure(fig, "11. Return quantity & refund amount theo Category × Return Reason", height=1250)
    fig.update_xaxes(title_text="Return Reason", row=1, col=1)
    fig.update_xaxes(title_text="Return Reason", row=2, col=1)
    fig.update_yaxes(title_text="Category", row=1, col=1)
    fig.update_yaxes(title_text="Category", row=2, col=1)
    fig.show()


## 12) Pie chart phân bổ rating theo Category và Segment

In [16]:

if reviews_df is None:
    print("Thiếu reviews.csv -> bỏ qua chart 12")
else:
    # 12a Category dropdown
    cat_list = sorted(reviews_df["category"].dropna().unique().tolist())
    fig_cat = go.Figure()

    for i, cat in enumerate(cat_list):
        tmp = reviews_df[reviews_df["category"] == cat].groupby("rating", as_index=False).size()
        fig_cat.add_trace(go.Pie(
            labels=tmp["rating"],
            values=tmp["size"],
            hole=0.45,
            name=str(cat),
            visible=(i == 0),
            textinfo="label+percent"
        ))

    buttons_cat = []
    for i, cat in enumerate(cat_list):
        vis = [False] * len(cat_list)
        vis[i] = True
        buttons_cat.append(dict(
            label=str(cat),
            method="update",
            args=[{"visible": vis}, {"title": f"12a. Rating distribution theo Category: {cat}"}]
        ))

    style_figure(fig_cat, f"12a. Rating distribution theo Category: {cat_list[0] if cat_list else 'N/A'}", height=760)
    fig_cat.update_layout(updatemenus=[dict(active=0, buttons=buttons_cat, x=1.08, y=1.0)])
    fig_cat.show()

    # 12b Segment dropdown
    seg_list = sorted(reviews_df["segment"].dropna().unique().tolist())
    fig_seg = go.Figure()

    for i, seg in enumerate(seg_list):
        tmp = reviews_df[reviews_df["segment"] == seg].groupby("rating", as_index=False).size()
        fig_seg.add_trace(go.Pie(
            labels=tmp["rating"],
            values=tmp["size"],
            hole=0.45,
            name=str(seg),
            visible=(i == 0),
            textinfo="label+percent"
        ))

    buttons_seg = []
    for i, seg in enumerate(seg_list):
        vis = [False] * len(seg_list)
        vis[i] = True
        buttons_seg.append(dict(
            label=str(seg),
            method="update",
            args=[{"visible": vis}, {"title": f"12b. Rating distribution theo Segment: {seg}"}]
        ))

    style_figure(fig_seg, f"12b. Rating distribution theo Segment: {seg_list[0] if seg_list else 'N/A'}", height=760)
    fig_seg.update_layout(updatemenus=[dict(active=0, buttons=buttons_seg, x=1.08, y=1.0)])
    fig_seg.show()


## 13) Xu hướng tổng tồn kho, nhập hàng và bán hàng theo tháng

In [17]:

if inventory is None:
    print("Thiếu inventory.csv -> bỏ qua chart 13")
else:
    inv_m = (
        inventory.groupby("snapshot_date", as_index=False)[["stock_on_hand", "units_received", "units_sold"]]
        .sum()
        .sort_values("snapshot_date")
    )

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=inv_m["snapshot_date"], y=inv_m["stock_on_hand"],
        mode="lines+markers", name="Stock on Hand",
        line=dict(width=MEDIUM_LINE), marker=dict(size=4),
        hovertemplate="Date: %{x}<br>Stock on Hand: %{y:,.0f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=inv_m["snapshot_date"], y=inv_m["units_received"],
        mode="lines+markers", name="Units Received",
        line=dict(width=MEDIUM_LINE), marker=dict(size=4),
        hovertemplate="Date: %{x}<br>Units Received: %{y:,.0f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=inv_m["snapshot_date"], y=inv_m["units_sold"],
        mode="lines+markers", name="Units Sold",
        line=dict(width=MEDIUM_LINE), marker=dict(size=4),
        hovertemplate="Date: %{x}<br>Units Sold: %{y:,.0f}<extra></extra>"
    ))

    style_figure(fig, "13. Xu hướng tổng tồn kho, nhập hàng và bán hàng theo tháng", height=760)
    fig.update_xaxes(title_text="Snapshot Date")
    fig.update_yaxes(title_text="Units")
    fig.show()


## 14) Heatmap stockout theo Category × Month

In [18]:

if inventory is None:
    print("Thiếu inventory.csv -> bỏ qua chart 14")
else:
    heat = inventory.groupby(["category", "month"], as_index=False)["stockout_days"].mean()
    heat["month_name"] = heat["month"].map(MONTH_NAME_MAP)
    pivot = heat.pivot(index="category", columns="month_name", values="stockout_days").fillna(0)
    pivot = pivot[ordered_month_cols(pivot.columns)]

    fig = px.imshow(
        pivot,
        aspect="auto",
        text_auto=".1f",
        color_continuous_scale="YlOrRd"
    )
    style_figure(fig, "14. Heatmap stockout theo Category × Month", height=720)
    fig.update_xaxes(title_text="Month")
    fig.update_yaxes(title_text="Category")
    fig.show()


## 15) Sell-through rate theo Category / Segment theo tháng

In [25]:

if inventory is None:
    print("Thiếu inventory.csv -> bỏ qua chart 15")
else:
    by_cat = inventory.groupby(["snapshot_date", "category"], as_index=False)["sell_through_rate"].mean()
    by_seg = inventory.groupby(["snapshot_date", "segment"], as_index=False)["sell_through_rate"].mean()

    fig = make_subplots(
        rows=2, cols=1,
        vertical_spacing=0.10,
        subplot_titles=("Sell-through rate theo Category", "Sell-through rate theo Segment")
    )

    for cat in sorted(by_cat["category"].dropna().unique()):
        tmp = by_cat[by_cat["category"] == cat]
        fig.add_trace(go.Scatter(
            x=tmp["snapshot_date"], y=tmp["sell_through_rate"],
            mode="lines+markers", name=f"Category: {cat}",
            line=dict(width=THIN_LINE), marker=dict(size=4),
            hovertemplate="Date: %{x}<br>STR: %{y:.2%}<extra></extra>"
        ), row=1, col=1)

    for seg in sorted(by_seg["segment"].dropna().unique()):
        tmp = by_seg[by_seg["segment"] == seg]
        fig.add_trace(go.Scatter(
            x=tmp["snapshot_date"], y=tmp["sell_through_rate"],
            mode="lines+markers", name=f"Segment: {seg}",
            line=dict(width=THIN_LINE), marker=dict(size=4),
            hovertemplate="Date: %{x}<br>STR: %{y:.2%}<extra></extra>"
        ), row=2, col=1)

    style_figure(fig, "15. Sell-through rate theo Category / Segment theo tháng", height=1300)
    fig.update_yaxes(title_text="Sell-through Rate", tickformat=".1%", row=1, col=1)
    fig.update_yaxes(title_text="Sell-through Rate", tickformat=".1%", row=2, col=1)
    fig.update_xaxes(title_text="Snapshot Date", row=2, col=1)
    fig.show()


## 16) 4 line delta charts: Units Received vs Units Sold cho 4 category lớn nhất

In [26]:

if inventory is None:
    print("Thiếu inventory.csv -> bỏ qua chart 16")
else:
    top4 = (
        inventory.groupby("category", as_index=False)["units_sold"]
        .sum()
        .sort_values("units_sold", ascending=False)
        .head(4)["category"]
        .tolist()
    )

    plot_df = (
        inventory[inventory["category"].isin(top4)]
        .groupby(["snapshot_date", "category"], as_index=False)[["units_received", "units_sold"]]
        .sum()
    )

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.05,
        subplot_titles=top4
    )

    for i, cat in enumerate(top4):
        tmp = plot_df[plot_df["category"] == cat].sort_values("snapshot_date")

        fig.add_trace(go.Scatter(
            x=tmp["snapshot_date"], y=tmp["units_received"],
            mode="lines", name=f"{cat} - Units Received",
            line=dict(width=THIN_LINE),
            hovertemplate="Date: %{x}<br>Units Received: %{y:,.0f}<extra></extra>"
        ), row=i+1, col=1)

        fig.add_trace(go.Scatter(
            x=tmp["snapshot_date"], y=tmp["units_sold"],
            mode="lines", name=f"{cat} - Units Sold",
            line=dict(width=THIN_LINE, dash="dash"),
            fill="tonexty",
            fillcolor="rgba(120,120,120,0.12)",
            hovertemplate="Date: %{x}<br>Units Sold: %{y:,.0f}<extra></extra>"
        ), row=i+1, col=1)

    style_figure(fig, "16. Units Received vs Units Sold cho 4 category lớn nhất", height=1800)
    fig.update_yaxes(title_text="Units")
    fig.update_xaxes(title_text="Snapshot Date")
    fig.show()


## 17) Stacked bar: doanh thu category chia theo Region / Age Group / Segment

In [21]:

fig = make_subplots(
    rows=3, cols=1,
    vertical_spacing=0.12,
    subplot_titles=(
        "17a. Revenue by Category stacked by Region",
        "17b. Revenue by Category stacked by Age Group",
        "17c. Revenue by Category stacked by Segment"
    )
)

if "region" in line_items.columns:
    reg_df = line_items.groupby(["category", "region"], as_index=False)["Revenue"].sum()
    for reg in sorted(reg_df["region"].dropna().unique()):
        tmp = reg_df[reg_df["region"] == reg]
        fig.add_trace(go.Bar(
            x=tmp["category"], y=tmp["Revenue"], name=f"Region: {reg}",
            hovertemplate="Category: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
        ), row=1, col=1)

if "age_group" in line_items.columns:
    age_df = line_items.groupby(["category", "age_group"], as_index=False)["Revenue"].sum()
    for age in sorted(age_df["age_group"].dropna().unique()):
        tmp = age_df[age_df["age_group"] == age]
        fig.add_trace(go.Bar(
            x=tmp["category"], y=tmp["Revenue"], name=f"Age Group: {age}",
            hovertemplate="Category: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
        ), row=2, col=1)

seg_df = line_items.groupby(["category", "segment"], as_index=False)["Revenue"].sum()
for seg in sorted(seg_df["segment"].dropna().unique()):
    tmp = seg_df[seg_df["segment"] == seg]
    fig.add_trace(go.Bar(
        x=tmp["category"], y=tmp["Revenue"], name=f"Segment: {seg}",
        hovertemplate="Category: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ), row=3, col=1)

fig.update_layout(barmode="stack")
style_figure(fig, "17. Stacked bar: doanh thu category chia theo Region / Age Group / Segment", height=1600)
fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f")
fig.update_xaxes(tickangle=-20)
fig.show()


## 18) Stacked Area Chart: phân bổ doanh thu theo Category và Segment theo thời gian

In [22]:

cat_ts = line_items.groupby(["month_start", "category"], as_index=False)["Revenue"].sum()
seg_ts = line_items.groupby(["month_start", "segment"], as_index=False)["Revenue"].sum()

fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.12,
    subplot_titles=("18a. Revenue mix theo Category", "18b. Revenue mix theo Segment")
)

for cat in sorted(cat_ts["category"].dropna().unique()):
    tmp = cat_ts[cat_ts["category"] == cat]
    fig.add_trace(go.Scatter(
        x=tmp["month_start"], y=tmp["Revenue"],
        stackgroup="one",
        mode="lines",
        name=f"Category: {cat}",
        line=dict(width=THIN_LINE),
        hovertemplate="Month: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ), row=1, col=1)

for seg in sorted(seg_ts["segment"].dropna().unique()):
    tmp = seg_ts[seg_ts["segment"] == seg]
    fig.add_trace(go.Scatter(
        x=tmp["month_start"], y=tmp["Revenue"],
        stackgroup="two",
        mode="lines",
        name=f"Segment: {seg}",
        line=dict(width=THIN_LINE),
        hovertemplate="Month: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ), row=2, col=1)

style_figure(fig, "18. Stacked Area Chart: phân bổ doanh thu theo Category / Segment theo thời gian", height=1350)
fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f")
fig.update_xaxes(title_text="Month")
fig.show()


## 19) Traffic metrics theo ngày + Revenue

In [23]:

if web is None:
    print("Thiếu web_traffic.csv -> bỏ qua chart 19")
else:
    rev_daily = sales_daily[["Date", "Revenue"]].rename(columns={"Date": "date"})
    traffic = web.merge(rev_daily, on="date", how="left").sort_values("date")

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(
            "Sessions theo ngày",
            "Unique Visitors theo ngày",
            "Page Views theo ngày",
            "Revenue theo ngày"
        )
    )

    fig.add_trace(go.Scatter(
        x=traffic["date"], y=traffic["sessions"],
        mode="lines", name="Sessions",
        line=dict(width=THIN_LINE),
        hovertemplate="Date: %{x}<br>Sessions: %{y:,.0f}<extra></extra>"
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=traffic["date"], y=traffic["unique_visitors"],
        mode="lines", name="Unique Visitors",
        line=dict(width=THIN_LINE),
        hovertemplate="Date: %{x}<br>Unique Visitors: %{y:,.0f}<extra></extra>"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=traffic["date"], y=traffic["page_views"],
        mode="lines", name="Page Views",
        line=dict(width=THIN_LINE),
        hovertemplate="Date: %{x}<br>Page Views: %{y:,.0f}<extra></extra>"
    ), row=3, col=1)

    fig.add_trace(go.Scatter(
        x=traffic["date"], y=traffic["Revenue"],
        mode="lines", name="Revenue",
        line=dict(width=THIN_LINE),
        hovertemplate="Date: %{x}<br>Revenue: %{y:,.0f} VNĐ<extra></extra>"
    ), row=4, col=1)

    style_figure(fig, "19. Traffic metrics theo ngày + Revenue", height=1700)
    fig.update_yaxes(title_text="Sessions", row=1, col=1)
    fig.update_yaxes(title_text="Unique Visitors", row=2, col=1)
    fig.update_yaxes(title_text="Page Views", row=3, col=1)
    fig.update_yaxes(title_text="Revenue (VNĐ)", tickformat=",.0f", row=4, col=1)
    fig.update_xaxes(title_text="Date", row=4, col=1)
    fig.show()



## Gợi ý dùng notebook này tốt nhất
- Chạy toàn bộ notebook để rà xem chart nào mạnh nhất với dữ liệu thật.
- Khi đưa vào report 4 trang, chỉ chọn khoảng **6–8 chart mạnh nhất**, không nên nhồi cả 19 chart.
- Có thể thêm:
  - annotation insight trực tiếp lên từng biểu đồ
  - caption 2–4 dòng dưới mỗi chart
  - export PNG / HTML để chèn vào report

Nếu muốn, bước tiếp theo tôi có thể làm tiếp:
1. một notebook **đã thêm caption insight mẫu dưới từng chart**, hoặc  
2. một notebook **lọc ra top 8 chart mạnh nhất cho report final**.
